# Zava data scientist mission notebook

This notebook is a starter artifact for the mission. It captures the workflow for schema discovery, corpus inspection, semantic retrieval, evaluation, and a brief model card.

## 1. Goal

Build a defensible semantic theme-discovery study over Zava customer text using the vector tool and SQL MCP evidence.

**Specifics:** 89-document corpus (44 reviews + 45 support chats) in English, Spanish, French. Go beyond "the tool returned neighbors" to a hand-labeled precision@k audit that shows *where and why* retrieval fails.

**Harness/model:** GitHub Copilot CLI, agent mode, SQL MCP tools. Model selection: `Auto`.

In [ ]:
import pandas as pd
from pathlib import Path

print('Notebook ready')

## 2. Turn-by-turn evidence log

Use this section to record the SQL MCP calls and outputs from each turn.

In [ ]:
turns = []

turns.append({
    'turn': 1,
    'prompt': "As Zava's Principal Data Scientist, discover the available entities and schema using the SQL MCP tools, then tell me the most relevant tables/fields for studying multilingual customer themes. Keep it grounded in actual tool output; if anything is unclear, say so.",
    'tool_calls': 'sqlMcpServer.describe_entities (x3) — FAILED to start',
    'result': "MCP runtime failed to start (address already in use). Agent fell back to reading .vscode/mcp.json + dab-config.json. Derived schema: Docs (DocId, SourceType, Body, Category, TagsJson, Embedding 1536), SupportChats, SupportTickets, Customers, FindSimilarDocsByDocId, + Products/SalesOrders for enrichment. Agent flagged that this was config-derived, not a live query.",
    'note': 'STRUGGLE: live discovery failed; agent honestly disclosed the fallback and recommended restarting the MCP server. MCP recovered by Turn 2.'
})

turns.append({
    'turn': 2,
    'prompt': "Before forming any hypothesis, show me counts or summaries that describe the corpus — document volume and the mix of sources, languages, and categories.",
    'tool_calls': 'sqlMcpServer.describe_entities + read_records (live)',
    'result': "Docs: 89 (44 Reviews / 45 Support Chats). SupportChats: 45. SupportTickets: 47. Customers: 1,228. SalesOrders: 1,800. SalesOrderLines: 2,747. Support-transcript languages: EN 28 / ES 13 / FR 4 (English-dominant, unbalanced).",
    'note': 'MCP live calls now working. Established corpus is modest but balanced review/chat.'
})

turns.append({
    'turn': 3,
    'prompt': "Pick one representative English seed and one Spanish seed, run the similarity tool for each, and explain emerging themes before evaluating quality.",
    'tool_calls': 'read_records (x4) + FindSimilarDocsByDocId on DocId 2 and DocId 9',
    'result': "English seed DocId 2 (negative) -> neighbors 7, 18, 21, 3 (quality/refund cluster). Spanish seed DocId 9 (positive) -> neighbors 15, 25, 18, 37 (fit/comfort cluster). DocId 18 appears in BOTH neighborhoods.",
    'note': 'Cross-language retrieval confirmed; DocId 18 overlap flagged for scrutiny.'
})

turns.append({
    'turn': 4,
    'prompt': "DocId 18 appears in both neighborhoods. Fetch its full Body and TagsJson. Is this a true cross-language thematic match, or does it appear due to shared surface vocabulary across two sentiments?",
    'tool_calls': 'read_records on DocId 18',
    'result': "DocId 18 is an ENGLISH POSITIVE review (rating:5). True match for the Spanish positive seed (good experience + minor defect); a topical-only false positive for the English negative seed (shares defect/fit vocabulary, opposite sentiment). Conclusion: model captures topic well, sentiment weakly.",
    'note': 'Core finding. Agent initially leaned toward calling it a clean cross-language theme; correction prompt reined it in.'
})

turns.append({
    'turn': 5,
    'prompt': "Show the cosine distances for the DocId 2 and DocId 9 neighbor lists.",
    'tool_calls': 'FindSimilarDocsByDocId (distances)',
    'result': "DocId 2: 7=0.2377, 18=0.3197, 21=0.3268, 3=0.3392. DocId 9: 15=0.2024, 25=0.2050, 18=0.2751, 37=0.3271. DocId 18 is closer to the positive seed (0.2751) than the negative seed (0.3197) — quantitative proof it belongs to the positive cluster.",
    'note': 'Distances objectively confirm the Turn 4 diagnosis.'
})

turns.append({
    'turn': 6,
    'prompt': "Label each DocId 2 neighbor RELEVANT/NOT RELEVANT for a negative-quality theme (relevant only if it shares both topic AND negative sentiment). Cover 7, 18, 21, 3. One short reason each.",
    'tool_calls': 'read_records (bodies) + manual labeling',
    'result': "7=RELEVANT (negative, refund), 18=NOT RELEVANT (positive sentiment), 21=RELEVANT (negative fit/quality), 3=RELEVANT (negative, refund). precision@4 = 0.75.",
    'note': 'First precision number. Single miss = DocId 18.'
})

turns.append({
    'turn': 7,
    'prompt': "Label each DocId 9 neighbor RELEVANT/NOT RELEVANT for a positive fit/comfort theme. Cover 15, 25, 18, 37. One short reason each.",
    'tool_calls': 'read_records (bodies) + manual labeling',
    'result': "15, 25, 18, 37 all RELEVANT. precision@4 = 1.00. Asymmetry: positive cluster retrieves cleanly (1.00) vs negative cluster (0.75) — retrieval is weakest exactly where the business cares most (complaints).",
    'note': 'Symmetric audit reveals the asymmetry finding.'
})

import pandas as pd
turn_log = pd.DataFrame(turns)
turn_log

## 3. Retrieval evaluation template

Fill in the label column manually after inspecting retrieved examples.

In [ ]:
evaluation = pd.DataFrame([
    # English NEGATIVE seed (DocId 2) — relevant = shares topic AND negative sentiment
    {'seed_doc_id': 2, 'seed_sentiment': 'negative', 'retrieved_doc_id': 7,  'cosine': 0.2377, 'label': 'RELEVANT',     'notes': 'Negative top review — poor fit, cheap fabric, worn stitching, refund demand'},
    {'seed_doc_id': 2, 'seed_sentiment': 'negative', 'retrieved_doc_id': 18, 'cosine': 0.3197, 'label': 'NOT RELEVANT', 'notes': 'POSITIVE review — praises fit/comfort; topical match only (false positive)'},
    {'seed_doc_id': 2, 'seed_sentiment': 'negative', 'retrieved_doc_id': 21, 'cosine': 0.3268, 'label': 'RELEVANT',     'notes': 'Negative — fit/comfort/quality complaints, did not match the price'},
    {'seed_doc_id': 2, 'seed_sentiment': 'negative', 'retrieved_doc_id': 3,  'cosine': 0.3392, 'label': 'RELEVANT',     'notes': 'Negative shorts — bad fit, cheap fabric, inconsistent sizing, full-refund request'},
    # Spanish POSITIVE seed (DocId 9) — relevant = shares topic AND positive sentiment
    {'seed_doc_id': 9, 'seed_sentiment': 'positive', 'retrieved_doc_id': 15, 'cosine': 0.2024, 'label': 'RELEVANT', 'notes': 'ES positive — comfortable fit, good sizing'},
    {'seed_doc_id': 9, 'seed_sentiment': 'positive', 'retrieved_doc_id': 25, 'cosine': 0.2050, 'label': 'RELEVANT', 'notes': 'ES positive — comfortable, fits well, minor defect tolerated'},
    {'seed_doc_id': 9, 'seed_sentiment': 'positive', 'retrieved_doc_id': 18, 'cosine': 0.2751, 'label': 'RELEVANT', 'notes': 'EN positive — fits well, comfortable, looks great (cross-language match)'},
    {'seed_doc_id': 9, 'seed_sentiment': 'positive', 'retrieved_doc_id': 37, 'cosine': 0.3271, 'label': 'RELEVANT', 'notes': 'ES positive — good fit, good fabric, comfortable all day'},
])

def precision_at_k(df, seed):
    sub = df[df['seed_doc_id'] == seed]
    return (sub['label'] == 'RELEVANT').sum() / len(sub)

print(f"precision@4 (DocId 2, negative): {precision_at_k(evaluation, 2):.2f}")
print(f"precision@4 (DocId 9, positive): {precision_at_k(evaluation, 9):.2f}")
evaluation

## 4. Theme summary

Write one paragraph summarizing the discovered themes, plus a short model-card note on limits and failure modes.

In [ ]:
theme_summary = """
Three themes ground the corpus, each traced through the vector tool:
1. Product-quality disappointment / refund (DocIds 2, 3, 7, 21) — negative reviews of premium
   items citing poor fit, cheap fabric, weak stitching, refund demands. HIGH confidence.
2. Fit & comfort satisfaction (DocIds 9, 15, 18, 37) — positive reviews praising fit, comfort,
   sizing. Crosses language (ES + EN). HIGH confidence.
3. Minor-defect tolerance (DocIds 18, 25 + seeds) — 'small defect but still great' — sits on the
   boundary; sentiment, not topic, decides which cluster it joins. MEDIUM confidence.
"""

model_card = """
MODEL CARD — Zava Voice-of-Customer semantic retrieval
Good for: cross-language topical retrieval; theme discovery without keyword rules; bridging
  reviews and support chats in one vector space.
NOT good for: sentiment-sensitive routing; automated refund/escalation decisions; statistical
  generalization on an 89-doc corpus.
Measured quality: precision@4 = 0.75 (negative seed, DocId 2) vs 1.00 (positive seed, DocId 9)
  over 8 hand-labeled pairs.
Key failure mode: captures TOPIC strongly, SENTIMENT weakly. DocId 18 (a positive review) is a
  correct neighbor for a positive seed but a false positive for a negative seed — distances
  confirm it (0.2751 positive vs 0.3197 negative). Retrieval is least trustworthy on complaints.
Limits: 89 docs; English-dominant (EN 28 / ES 13 / FR 4 in support transcripts); no ground truth.
Next steps: add a sentiment dimension/filter; build a labeled gold set; balance & expand corpus;
  keep refund/escalation decisions human-in-the-loop.
Process note: live MCP describe_entities failed on Turn 1 (port in use); agent disclosed the
  config-file fallback rather than bluffing, and live SQL calls recovered from Turn 2 onward.
"""

print(theme_summary)
print(model_card)